# News-to-Social NER Domain Adaptation

This notebook is a simplified, notebook-first version of the project.

It keeps only the main ideas needed for the case study:
- source-only baseline
- few-shot adaptation
- normalization
- self-training
- strict NER evaluation

The task is **NER only**. We transfer a news-trained NER model to WNUT17 social-media text and keep only the overlapping labels `PER`, `LOC`, and `ORG`.

## 1. Pipeline Overview

The notebook follows one simple pipeline:

1. Load the source checkpoint and WNUT17.
2. Map WNUT17 labels into the reduced label space.
3. Build a small few-shot training subset.
4. Optionally normalize noisy social-media tokens.
5. Fine-tune on the few-shot target data.
6. Optionally generate pseudo-labels on the remaining target data.
7. Re-train on gold + pseudo-labeled data.
8. Evaluate with strict entity-level F1.

These steps define the domain-adaptation pipeline used in this notebook.

In [1]:
from __future__ import annotations

import random
import re
from collections import Counter
from pathlib import Path
from typing import Any

import evaluate
import numpy as np
import pandas as pd
import torch
from datasets import Dataset, load_dataset, concatenate_datasets
from seqeval.scheme import IOB2
from transformers import (
    AutoConfig,
    AutoModelForTokenClassification,
    AutoTokenizer,
    DataCollatorForTokenClassification,
    Trainer,
    TrainingArguments,
)


c:\Users\Dinar\miniconda3\envs\machine-learning\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Fixed Experiment Settings

The main experiment settings are fixed here for clarity and reproducibility.

In [2]:
# Minimal output directory required by Hugging Face Trainer.
TRAINER_OUTPUT_DIR = "tmp_trainer"

# Source model trained on CoNLL2003 news NER.
MODEL_NAME = "andi611/roberta-base-ner-conll2003"

# Target-domain dataset.
DATASET_NAME = "flaitenberger/wnut_17"

# Stronger reference model already fine-tuned on WNUT.
BENCHMARK_MODEL_NAME = "emilys/twitter-roberta-base-WNUT"

# Fixed seed for reproducibility.
SEED = 5150

# Maximum tokenized sequence length.
MAX_LENGTH = 128

# Main optimizer settings.
LEARNING_RATE = 3e-5
WEIGHT_DECAY = 0.01

# Number of epochs for the few-shot stage.
NUM_EPOCHS = 10

# Number of epochs for the self-training stage.
SELF_TRAIN_EPOCHS = 10

# Batch size for both training and evaluation.
BATCH_SIZE = 8

# Minimum confidence required to keep a pseudo-labeled sentence.
PSEUDO_LABEL_THRESHOLD = 0.98

# Maximum pseudo-labeled set size relative to the gold few-shot set.
PSEUDO_LABEL_MAX_MULTIPLIER = 1

# Use mixed precision if CUDA is available.
USE_FP16 = torch.cuda.is_available()

# Reduced label space used throughout the notebook.
TARGET_LABELS = ["O", "B-PER", "I-PER", "B-LOC", "I-LOC", "B-ORG", "I-ORG"]

# Map label name -> integer id.
LABEL2ID = {label: idx for idx, label in enumerate(TARGET_LABELS)}

# Map integer id -> label name.
ID2LABEL = {idx: label for label, idx in LABEL2ID.items()}

# Keep only overlapping WNUT17 entity types.
WNUT_TO_TARGET = {
    "B-person": "B-PER",
    "I-person": "I-PER",
    "B-location": "B-LOC",
    "I-location": "I-LOC",
    "B-corporation": "B-ORG",
    "I-corporation": "I-ORG",
}

# Print basic environment information.
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Source model:", MODEL_NAME)
print("Benchmark model:", BENCHMARK_MODEL_NAME)


CUDA available: True
GPU: NVIDIA GeForce RTX 3050 Ti Laptop GPU
Source model: andi611/roberta-base-ner-conll2003
Benchmark model: emilys/twitter-roberta-base-WNUT


## 3. Label Mapping and Normalization

We keep only the overlapping labels:
- `person -> PER`
- `location -> LOC`
- `corporation -> ORG`

All other WNUT17 entity types are ignored.

Normalization is intentionally lightweight:
- usernames become `@USER`
- URLs become `HTTPURL`
- token boundaries are preserved

In [3]:
# Match social-media usernames such as @john_smith.
USERNAME_RE = re.compile(r"^@[A-Za-z0-9_]+$")

# Match URLs such as http://... or www....
URL_RE = re.compile(r"^(https?://\S+|www\.\S+)$", flags=re.IGNORECASE)

# Recover coarse entity type from reduced non-O labels.
ENTITY_TYPE_BY_ID = {
    label_id: label_name.split("-", maxsplit=1)[1]
    for label_name, label_id in LABEL2ID.items()
    if label_name != "O"
}


def set_seed(seed: int) -> None:
    # Fix Python randomness.
    random.seed(seed)
    # Fix NumPy randomness.
    np.random.seed(seed)
    # Fix PyTorch randomness on CPU.
    torch.manual_seed(seed)
    # Fix PyTorch randomness on GPU if available.
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def map_label_name_to_target_id(label_name: str) -> int:
    # Keep the outside tag as is.
    if label_name == "O":
        return LABEL2ID["O"]
    # Map WNUT tag into the reduced label space.
    target_name = WNUT_TO_TARGET.get(label_name)
    # Ignore unsupported WNUT labels.
    if target_name is None:
        return -100
    # Return the reduced integer id.
    return LABEL2ID[target_name]


def normalize_tokens(tokens: list[str]) -> list[str]:
    # Store normalized tokens here.
    normalized = []
    # Process tokens one by one.
    for token in tokens:
        # Replace usernames with a canonical token.
        if USERNAME_RE.match(token):
            normalized.append("@USER")
        # Replace URLs with a canonical token.
        elif URL_RE.match(token):
            normalized.append("HTTPURL")
        # Leave other tokens unchanged.
        else:
            normalized.append(token)
    # Return normalized token list.
    return normalized


def is_social_noise_token(token: str) -> bool:
    # Heuristic flag used in lightweight error analysis.
    return bool(USERNAME_RE.match(token) or URL_RE.match(token) or token.startswith("#"))


## 4. Load and Prepare WNUT17

This section:
- loads WNUT17
- converts its labels into the reduced label space
- builds a deterministic few-shot subset
- optionally applies normalization

The few-shot sampler is simple and deterministic: it prefers sentences that contain entities, especially `ORG` mentions.

In [4]:
def convert_split_to_reduced_labels(split_dataset: Dataset, split_name: str) -> Dataset:
    # Original WNUT tag names stored in the dataset feature definition.
    label_names = split_dataset.features["ner_tags"].feature.names

    def convert_example(example: dict[str, Any], index: int) -> dict[str, Any]:
        # Convert each original WNUT tag into the reduced label space.
        reduced_tags = [map_label_name_to_target_id(label_names[tag]) for tag in example["ner_tags"]]
        # Copy tokens so we can keep both original and model-input versions.
        tokens = list(example["tokens"])
        # Return a simpler schema for the rest of the notebook.
        return {
            # Deterministic example id.
            "example_id": f"{split_name}-{index}",
            # Current token sequence used by the model.
            "tokens": tokens,
            # Original token sequence for later inspection.
            "original_tokens": tokens,
            # Reduced token-level labels.
            "target_ner_tags": reduced_tags,
        }

    # Apply conversion to the whole split.
    return split_dataset.map(
        convert_example,
        with_indices=True,
        remove_columns=split_dataset.column_names,
    )


def count_entities(tags: list[int]) -> tuple[int, int]:
    # Total number of entity spans in the sentence.
    total_entities = 0
    # Number of ORG spans in the sentence.
    org_entities = 0
    # Track previous tag to detect span starts.
    previous_tag = 0
    # Iterate over token-level labels.
    for tag_id in tags:
        # Reset span tracking on O or ignored labels.
        if tag_id <= 0:
            previous_tag = 0
            continue
        # Start a new entity on B-tags or broken I-tag continuity.
        is_start = tag_id % 2 == 1 or tag_id != previous_tag + 1
        if is_start:
            # Count a new entity span.
            total_entities += 1
            # Count ORG spans separately because they are harder.
            if ENTITY_TYPE_BY_ID[int(tag_id)] == "ORG":
                org_entities += 1
        # Update previous tag.
        previous_tag = tag_id
    # Return both counts.
    return total_entities, org_entities


def sample_fewshot_sentences(train_dataset: Dataset, shot_count: int, seed: int) -> tuple[Dataset, Dataset]:
    # Source-only uses no target-domain labeled sentences.
    if shot_count == 0:
        return train_dataset.select([]), train_dataset

    # Local RNG so sampling stays deterministic.
    rng = random.Random(seed)
    # Start from all sentence indices.
    indices = list(range(len(train_dataset)))
    # Shuffle before ranking to make tie-breaking reproducible.
    rng.shuffle(indices)

    # Store ranking tuples here.
    ranked = []
    # Score each sentence by how useful it is for few-shot NER.
    for index in indices:
        # Count total entities and ORG entities.
        entity_count, org_count = count_entities(train_dataset[index]["target_ner_tags"])
        # Higher ORG count and higher entity count should be preferred.
        ranked.append((org_count, entity_count, index))

    # Sort so the best candidate sentences come first.
    ranked.sort(reverse=True)
    # Keep only the requested number of few-shot sentences.
    selected_indices = sorted(index for _, _, index in ranked[:shot_count])
    # Convert to a set for fast membership checks.
    selected_set = set(selected_indices)
    # Everything not selected is treated as unlabeled target data.
    remaining_indices = [i for i in range(len(train_dataset)) if i not in selected_set]
    # Return gold few-shot data and remaining unlabeled data.
    return train_dataset.select(selected_indices), train_dataset.select(remaining_indices)


def maybe_normalize_dataset(split_dataset: Dataset, enabled: bool) -> Dataset:
    # Skip normalization if the experiment does not use it.
    if not enabled:
        return split_dataset
    # Replace noisy tokens with canonical forms.
    return split_dataset.map(lambda row: {"tokens": normalize_tokens(row["tokens"])})


def tokenize_and_align_labels(examples: dict[str, list[Any]], tokenizer: Any) -> dict[str, Any]:
    # Tokenize already split word tokens.
    tokenized = tokenizer(
        examples["tokens"],
        is_split_into_words=True,
        truncation=True,
        max_length=MAX_LENGTH,
    )

    # Store aligned labels here.
    aligned_labels = []
    # Process each sentence in the batch.
    for batch_index, word_labels in enumerate(examples["target_ner_tags"]):
        # Recover mapping from subtokens to original word indices.
        word_ids = tokenized.word_ids(batch_index=batch_index)
        # Track previous word id to detect first subtoken.
        previous_word_id = None
        # Store aligned labels for this sentence.
        labels = []
        # Iterate over tokenized positions.
        for word_id in word_ids:
            # Ignore special tokens.
            if word_id is None:
                labels.append(-100)
            # Keep label only on the first subtoken of each word.
            elif word_id != previous_word_id:
                labels.append(int(word_labels[word_id]))
            # Ignore later subtokens of the same word.
            else:
                labels.append(-100)
            # Update the tracker.
            previous_word_id = word_id
        # Save the aligned label sequence.
        aligned_labels.append(labels)

    # Attach aligned labels to tokenized output.
    tokenized["labels"] = aligned_labels
    return tokenized


def tokenize_dataset(split_dataset: Dataset, tokenizer: Any) -> Dataset:
    # Tokenize the whole split and remove original raw columns.
    return split_dataset.map(
        lambda batch: tokenize_and_align_labels(batch, tokenizer),
        batched=True,
        remove_columns=split_dataset.column_names,
    )


def count_tokens(split_dataset: Dataset) -> Counter[str]:
    # Token frequency counter used for simple rare-token heuristics.
    counter = Counter()
    # Iterate over sentences.
    for tokens in split_dataset["tokens"]:
        # Iterate over tokens inside each sentence.
        for token in tokens:
            # Lowercase to reduce sparsity in counts.
            counter[token.lower()] += 1
    # Return token counts.
    return counter


# Fix randomness before dataset processing.
set_seed(SEED)
# Load the target-domain dataset.
raw_datasets = load_dataset(DATASET_NAME)
# Convert target train split to reduced labels.
train_raw = convert_split_to_reduced_labels(raw_datasets["train"], "train")
# Convert target validation split to reduced labels.
validation_raw = convert_split_to_reduced_labels(raw_datasets["validation"], "validation")
# Convert target test split to reduced labels.
test_raw = convert_split_to_reduced_labels(raw_datasets["test"], "test")
# Build token counts from target train for lightweight error analysis.
token_counts = count_tokens(train_raw)

# Print split sizes.
print(len(train_raw), len(validation_raw), len(test_raw))


Repo card metadata block was not found. Setting CardData to empty.
Map: 100%|██████████| 1287/1287 [00:00<00:00, 7857.49 examples/s]


3394 1009 1287


## 5. Load the Source Model

The source checkpoint is now a RoBERTa-base model trained on CoNLL2003 news NER. We adapt its classifier head to the reduced 7-label output space used in this notebook.

In [5]:
def get_classifier_layer(model: Any) -> torch.nn.Module:
    # Some token classification models call the output layer `classifier`.
    for name in ("classifier", "score"):
        # Try to fetch the layer by name.
        layer = getattr(model, name, None)
        # Stop once the layer is found.
        if layer is not None:
            return layer
    # Raise an explicit error if the architecture is unexpected.
    raise AttributeError("Classifier layer not found.")


def load_model_and_tokenizer() -> tuple[Any, Any]:
    # Load the tokenizer that belongs to the source checkpoint.
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

    # Build a new config for the reduced target label space.
    config = AutoConfig.from_pretrained(
        MODEL_NAME,
        num_labels=len(TARGET_LABELS),
        id2label=ID2LABEL,
        label2id=LABEL2ID,
    )
    # Load the source model with a resized token classification head.
    model = AutoModelForTokenClassification.from_pretrained(
        MODEL_NAME,
        config=config,
        ignore_mismatched_sizes=True,
    )

    # Load the original checkpoint again so we can copy useful classifier rows.
    source_model = AutoModelForTokenClassification.from_pretrained(MODEL_NAME)
    # Get the original classifier layer.
    source_classifier = get_classifier_layer(source_model)
    # Get the resized target classifier layer.
    target_classifier = get_classifier_layer(model)

    # Hardcoded CoNLL label order for this checkpoint.
    # This is necessary because the config label names are generic LABEL_0 ... LABEL_8.
    source_label_order = {
        "O": 0,
        "B-PER": 1,
        "I-PER": 2,
        "B-ORG": 3,
        "I-ORG": 4,
        "B-LOC": 5,
        "I-LOC": 6,
    }

    # Copy source classifier rows for overlapping labels.
    with torch.no_grad():
        # Iterate over reduced target labels that exist in the source task.
        for label_name, source_idx in source_label_order.items():
            # Get target row index for this label.
            target_idx = LABEL2ID[label_name]
            # Copy source classifier weights into the reduced head.
            target_classifier.weight[target_idx].copy_(source_classifier.weight[source_idx])
            # Copy source classifier bias into the reduced head.
            target_classifier.bias[target_idx].copy_(source_classifier.bias[source_idx])

    # Return the adapted model and tokenizer.
    return model, tokenizer


# Quick sanity check that model loading works.
model, tokenizer = load_model_and_tokenizer()
print(type(model).__name__)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 16109.52it/s]
RobertaForTokenClassification LOAD REPORT from: andi611/roberta-base-ner-conll2003
Key                             | Status     |                                                                                       
--------------------------------+------------+---------------------------------------------------------------------------------------
roberta.embeddings.position_ids | UNEXPECTED |                                                                                       
classifier.weight               | MISMATCH   | Reinit due to size mismatch - ckpt: torch.Size([9, 768]) vs model:torch.Size([7, 768])
classifier.bias                 | MISMATCH   | Reinit due to size mismatch - ckpt: torch.Size([9]) vs model:torch.Size([7])          

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISMATCH	:ckpt weights were loaded, but they did not m

RobertaForTokenClassification


## 6. Evaluation and Error Analysis

We use strict entity-level `seqeval` metrics. A prediction is correct only if both the entity span and the entity type match the gold annotation.

In [6]:
# Load strict NER evaluation metric.
seqeval_metric = evaluate.load("seqeval")


def decode_predictions(predictions: np.ndarray, labels: np.ndarray) -> tuple[list[list[str]], list[list[str]]]:
    # Trainer returns 3D logits, while the benchmark helper returns 2D label ids.
    predicted_ids = predictions.argmax(axis=2) if predictions.ndim == 3 else predictions
    # Store decoded prediction label strings.
    decoded_predictions = []
    # Store decoded gold label strings.
    decoded_labels = []
    # Process one sentence at a time.
    for prediction_row, label_row in zip(predicted_ids, labels):
        # Sentence-level prediction labels.
        sentence_predictions = []
        # Sentence-level gold labels.
        sentence_labels = []
        # Process token positions.
        for predicted_id, gold_id in zip(prediction_row, label_row):
            # Skip ignored positions such as subtokens and special tokens.
            if int(gold_id) == -100:
                continue
            # Convert predicted label id into string label.
            sentence_predictions.append(ID2LABEL[int(predicted_id)])
            # Convert gold label id into string label.
            sentence_labels.append(ID2LABEL[int(gold_id)])
        # Save decoded sentence predictions.
        decoded_predictions.append(sentence_predictions)
        # Save decoded sentence labels.
        decoded_labels.append(sentence_labels)
    # Return decoded label sequences.
    return decoded_predictions, decoded_labels


def compute_metrics_from_arrays(predictions: np.ndarray, labels: np.ndarray) -> dict[str, float]:
    # Convert ids/logits into string labels required by seqeval.
    decoded_predictions, decoded_labels = decode_predictions(predictions, labels)
    # Compute strict entity-level metrics.
    results = seqeval_metric.compute(
        predictions=decoded_predictions,
        references=decoded_labels,
        mode="strict",
        scheme="IOB2",
    )
    # Return the metrics we care about.
    return {
        "precision": float(results["overall_precision"]),
        "recall": float(results["overall_recall"]),
        "f1": float(results["overall_f1"]),
        "accuracy": float(results["overall_accuracy"]),
    }


def trainer_compute_metrics(eval_prediction):
    # Unpack predictions and labels from the Trainer wrapper object.
    predictions, labels = eval_prediction
    # Reuse the main metric function.
    return compute_metrics_from_arrays(predictions, labels)


def simple_error_table(raw_dataset: Dataset, predictions: np.ndarray, labels: np.ndarray, max_rows: int = 20) -> pd.DataFrame:
    # Decode predictions and labels into strings.
    decoded_predictions, decoded_labels = decode_predictions(predictions, labels)
    # Store error rows here.
    rows = []
    # Compare one sentence at a time.
    for index, (predicted_labels, gold_labels) in enumerate(zip(decoded_predictions, decoded_labels)):
        # Skip correct predictions.
        if predicted_labels == gold_labels:
            continue
        # Get the model-input tokens for this example.
        tokens = raw_dataset[index]["tokens"]
        # Keep only tokens where gold and prediction differ.
        mismatch_tokens = [token for token, gold, pred in zip(tokens, gold_labels, predicted_labels) if gold != pred]
        # Build one readable error row.
        rows.append(
            {
                # Example id for reference.
                "example_id": raw_dataset[index]["example_id"],
                # Original sentence text.
                "tokens": " ".join(raw_dataset[index]["original_tokens"]),
                # Gold label sequence.
                "gold_labels": " ".join(gold_labels),
                # Predicted label sequence.
                "predicted_labels": " ".join(predicted_labels),
                # Tokens where the model made mistakes.
                "mismatch_tokens": " ".join(mismatch_tokens),
                # Heuristic flag for social-media noise.
                "has_social_noise": any(is_social_noise_token(token) for token in mismatch_tokens),
                # Heuristic flag for rare tokens.
                "has_rare_token": any(token_counts.get(token.lower(), 0) <= 1 for token in mismatch_tokens),
            }
        )
    # Return a small error sample.
    return pd.DataFrame(rows[:max_rows])


## 7. Training Helpers

There are only two training stages in this notebook:
- few-shot fine-tuning on gold labels
- optional self-training on gold + pseudo-labeled examples

In [7]:
def build_training_arguments(run_name: str, num_epochs: int) -> TrainingArguments:
    # Build shared Trainer arguments used in all fine-tuning runs.
    return TrainingArguments(
        # Minimal directory required by Trainer.
        output_dir=TRAINER_OUTPUT_DIR,
        # Optimizer learning rate.
        learning_rate=LEARNING_RATE,
        # Weight decay regularization.
        weight_decay=WEIGHT_DECAY,
        # Fixed number of training epochs.
        num_train_epochs=float(num_epochs),
        # Per-device training batch size.
        per_device_train_batch_size=BATCH_SIZE,
        # Per-device evaluation batch size.
        per_device_eval_batch_size=BATCH_SIZE,
        # Run validation every epoch.
        eval_strategy="epoch",
        # Save checkpoints every epoch.
        save_strategy="epoch",
        # Reload the best checkpoint according to validation F1.
        load_best_model_at_end=True,
        # Metric used to select the best checkpoint.
        metric_for_best_model="eval_f1",
        # Higher F1 is better.
        greater_is_better=True,
        # Log once per epoch.
        logging_strategy="epoch",
        # Keep only a couple of checkpoints.
        save_total_limit=2,
        # Use fp16 if possible.
        fp16=USE_FP16,
        # Disable external reporting integrations.
        report_to=[],
        # Fix training seed.
        seed=SEED,
        # Fix data shuffling seed.
        data_seed=SEED,
    )


def train_model(model: Any, tokenizer: Any, train_dataset: Dataset, validation_dataset: Dataset, run_name: str, num_epochs: int) -> Any:
    # Build a Trainer for token classification fine-tuning.
    trainer = Trainer(
        # Current model.
        model=model,
        # Training arguments.
        args=build_training_arguments(run_name, num_epochs),
        # Gold training dataset.
        train_dataset=train_dataset,
        # Validation dataset used for checkpoint selection.
        eval_dataset=validation_dataset,
        # Dynamic padding collator for token classification.
        data_collator=DataCollatorForTokenClassification(tokenizer),
        # Strict entity-level evaluation metrics.
        compute_metrics=trainer_compute_metrics,
        # Tokenizer for processing metadata.
        processing_class=tokenizer,
    )
    # Run training.
    trainer.train()
    # Return the best model loaded by Trainer.
    return trainer.model


def evaluate_model(model: Any, tokenizer: Any, dataset: Dataset) -> tuple[dict[str, float], Any]:
    # Build a lightweight Trainer just for prediction/evaluation.
    trainer = Trainer(
        # Current model.
        model=model,
        # Minimal evaluation arguments.
        args=TrainingArguments(
            # Minimal output directory.
            output_dir=TRAINER_OUTPUT_DIR,
            # Evaluation batch size.
            per_device_eval_batch_size=BATCH_SIZE,
            # Disable external reporting integrations.
            report_to=[],
        ),
        # Token classification collator.
        data_collator=DataCollatorForTokenClassification(tokenizer),
        # Reuse strict entity-level metrics.
        compute_metrics=trainer_compute_metrics,
        # Tokenizer for processing metadata.
        processing_class=tokenizer,
    )
    # Run prediction on the split.
    prediction_output = trainer.predict(dataset)
    # Recompute metrics explicitly from raw predictions.
    metrics = compute_metrics_from_arrays(prediction_output.predictions, prediction_output.label_ids)
    # Return both metrics and raw prediction object.
    return metrics, prediction_output


## 8. Self-Training

The self-training idea is simple:
- predict labels on the remaining unlabeled target sentences
- keep only confident sentences with at least one non-`O` entity
- merge them with the few-shot gold data
- train again

In [8]:
def build_empty_pseudo_dataset() -> Dataset:
    # Build an empty dataset with the same schema as a pseudo-labeled dataset.
    return Dataset.from_dict(
        {
            # Example ids.
            "example_id": [],
            # Model-input tokens.
            "tokens": [],
            # Original tokens.
            "original_tokens": [],
            # Predicted pseudo-labels.
            "target_ner_tags": [],
        }
    )


def generate_pseudo_labels(raw_dataset: Dataset, model: Any, tokenizer: Any, max_samples: int) -> tuple[Dataset, pd.DataFrame]:
    # Return early if there is no unlabeled data.
    if len(raw_dataset) == 0:
        return build_empty_pseudo_dataset(), pd.DataFrame()

    # Get current device from the model.
    device = next(model.parameters()).device
    # Switch model to evaluation mode.
    model.eval()
    # Pseudo-labeled rows that survive filtering.
    kept_rows = []
    # Audit table for inspection.
    audit_rows = []

    # Process one unlabeled sentence at a time.
    for row in raw_dataset:
        # Tokenize the sentence as split words.
        encoded = tokenizer(
            row["tokens"],
            is_split_into_words=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt",
        )
        # Recover mapping from subtokens to original word ids.
        word_ids = encoded.word_ids(batch_index=0)
        # Move tensors to the model device.
        encoded = {key: value.to(device) for key, value in encoded.items()}

        # Run inference without gradient tracking.
        with torch.no_grad():
            logits = model(**encoded).logits[0]
            # Convert logits to probabilities.
            probabilities = torch.softmax(logits, dim=-1).cpu().numpy()

        # Predicted word-level labels.
        predicted_tags = []
        # Confidence scores only for predicted entity tokens.
        entity_confidences = []
        # Track which words were already processed.
        seen_word_ids = set()
        # Iterate over subtoken positions.
        for token_index, word_id in enumerate(word_ids):
            # Skip special tokens and later subtokens of the same word.
            if word_id is None or word_id in seen_word_ids:
                continue
            # Mark this word as processed.
            seen_word_ids.add(word_id)
            # Pick the most likely label at this token position.
            label_id = int(np.argmax(probabilities[token_index]))
            # Read its confidence.
            confidence = float(probabilities[token_index][label_id])
            # Save predicted word label.
            predicted_tags.append(label_id)
            # Keep confidence only for entity predictions.
            if label_id != 0:
                entity_confidences.append(confidence)

        # Skip examples where alignment failed.
        if len(predicted_tags) != len(row["tokens"]):
            continue

        # Keep only sentences where the model predicted at least one entity.
        has_entity = any(label_id != 0 for label_id in predicted_tags)
        # Average confidence only across entity tokens.
        confidence = float(np.mean(entity_confidences)) if entity_confidences else 0.0
        # Final keep decision.
        keep = has_entity and confidence >= PSEUDO_LABEL_THRESHOLD

        # Save audit info for inspection.
        audit_rows.append(
            {
                # Original example id.
                "example_id": row["example_id"],
                # Average entity-token confidence.
                "confidence": confidence,
                # Whether at least one entity was predicted.
                "has_entity_prediction": has_entity,
                # Whether the example passed the filter.
                "kept": keep,
            }
        )

        # Store the pseudo-labeled example if it passed filtering.
        if keep:
            kept_rows.append(
                {
                    # Mark example as pseudo-labeled.
                    "example_id": f"pseudo-{row['example_id']}",
                    # Current model-input tokens.
                    "tokens": row["tokens"],
                    # Original raw tokens.
                    "original_tokens": row["original_tokens"],
                    # Predicted pseudo-label sequence.
                    "target_ner_tags": predicted_tags,
                    # Confidence used for ranking.
                    "confidence": confidence,
                }
            )

    # Rank pseudo-labeled rows by confidence.
    kept_rows.sort(key=lambda row: row["confidence"], reverse=True)
    # Keep only the top pseudo-labeled rows.
    kept_rows = kept_rows[:max_samples]

    # Return an empty dataset if nothing survived.
    if not kept_rows:
        return build_empty_pseudo_dataset(), pd.DataFrame(audit_rows)

    # Convert kept pseudo-labeled rows back into a Dataset object.
    pseudo_dataset = Dataset.from_dict(
        {
            # Example ids.
            "example_id": [row["example_id"] for row in kept_rows],
            # Model-input tokens.
            "tokens": [row["tokens"] for row in kept_rows],
            # Original tokens.
            "original_tokens": [row["original_tokens"] for row in kept_rows],
            # Predicted pseudo-label sequences.
            "target_ner_tags": [row["target_ner_tags"] for row in kept_rows],
        }
    )
    # Return pseudo-labeled dataset and audit table.
    return pseudo_dataset, pd.DataFrame(audit_rows)


## 9. One Function to Run an Experiment

This is the only high-level function in the notebook. It runs one of the four cases:
- source-only
- few-shot
- few-shot + normalization
- few-shot + self-training

In [9]:
def run_experiment(name: str, shot_count: int, do_normalization: bool, do_self_training: bool) -> dict[str, Any]:
    # Reset randomness before every run.
    set_seed(SEED)
    # Load a fresh source model for this experiment.
    model, tokenizer = load_model_and_tokenizer()

    # Apply the same normalization policy to all target-domain splits.
    train_split = maybe_normalize_dataset(train_raw, do_normalization)
    validation_split = maybe_normalize_dataset(validation_raw, do_normalization)
    test_split = maybe_normalize_dataset(test_raw, do_normalization)

    # Build few-shot gold data and remaining unlabeled target data.
    fewshot_train_raw, unlabeled_raw = sample_fewshot_sentences(train_split, shot_count, SEED)
    # Tokenize validation split.
    validation_tokenized = tokenize_dataset(validation_split, tokenizer)
    # Tokenize test split.
    test_tokenized = tokenize_dataset(test_split, tokenizer)

    # Source-only skips target-domain fine-tuning.
    if shot_count > 0:
        # Tokenize few-shot gold training data.
        train_tokenized = tokenize_dataset(fewshot_train_raw, tokenizer)
        # Fine-tune on gold few-shot labels.
        model = train_model(model, tokenizer, train_tokenized, validation_tokenized, f"{name}_train", NUM_EPOCHS)

    # Default pseudo-label count.
    pseudo_count = 0
    # Self-training is applied only after an initial few-shot model exists.
    if do_self_training and shot_count > 0:
        # Set the cap on pseudo-labeled data size.
        max_pseudo = shot_count * PSEUDO_LABEL_MAX_MULTIPLIER
        # Generate pseudo-labels from the current model.
        pseudo_dataset, pseudo_audit = generate_pseudo_labels(unlabeled_raw, model, tokenizer, max_pseudo)
        # Store the number of kept pseudo-labeled examples.
        pseudo_count = len(pseudo_dataset)
        # Continue only if some pseudo-labeled examples survived filtering.
        if len(pseudo_dataset) > 0:
            # Merge gold and pseudo-labeled data.
            merged_train = concatenate_datasets([fewshot_train_raw, pseudo_dataset])
            # Tokenize merged training data.
            merged_train_tokenized = tokenize_dataset(merged_train, tokenizer)
            # Fine-tune again on the merged set.
            model = train_model(model, tokenizer, merged_train_tokenized, validation_tokenized, f"{name}_selftrain", SELF_TRAIN_EPOCHS)

    # Evaluate final model on validation.
    validation_metrics, _ = evaluate_model(model, tokenizer, validation_tokenized)
    # Evaluate final model on test.
    test_metrics, test_prediction = evaluate_model(model, tokenizer, test_tokenized)
    # Build a small qualitative error sample.
    error_table = simple_error_table(test_split, test_prediction.predictions, test_prediction.label_ids)

    # Return the main summary metrics for this experiment.
    return {
        # Experiment name.
        "experiment": name,
        # Number of gold target-domain training examples.
        "shot_count": shot_count,
        # Whether normalization was enabled.
        "do_normalization": do_normalization,
        # Whether self-training was enabled.
        "do_self_training": do_self_training,
        # Size of the gold few-shot training set.
        "train_examples": len(fewshot_train_raw),
        # Size of the retained pseudo-labeled set.
        "pseudo_examples": pseudo_count,
        # Validation F1.
        "validation_f1": validation_metrics["f1"],
        # Test F1.
        "test_f1": test_metrics["f1"],
    }


## 10. Smoke Test

Run one small experiment first to make sure everything works.

In [10]:
smoke_result = run_experiment(
    name="adapt_250",
    shot_count=250,
    do_normalization=False,
    do_self_training=False,
)
pd.DataFrame([smoke_result])


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 17196.50it/s]
RobertaForTokenClassification LOAD REPORT from: andi611/roberta-base-ner-conll2003
Key                             | Status     |                                                                                       
--------------------------------+------------+---------------------------------------------------------------------------------------
roberta.embeddings.position_ids | UNEXPECTED |                                                                                       
classifier.weight               | MISMATCH   | Reinit due to size mismatch - ckpt: torch.Size([9, 768]) vs model:torch.Size([7, 768])
classifier.bias                 | MISMATCH   | Reinit due to size mismatch - ckpt: torch.Size([9]) vs model:torch.Size([7])          

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISMATCH	:ckpt weights were loaded, but they did not m

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.183691,0.094617,0.676516,0.752595,0.712531,0.978125
2,0.093299,0.084979,0.719472,0.754325,0.736486,0.980227
3,0.051757,0.093402,0.762653,0.756055,0.759340,0.982264
4,0.026189,0.099798,0.728707,0.799308,0.762376,0.981344
5,0.015369,0.119224,0.724307,0.768166,0.745592,0.980621
6,0.014905,0.119679,0.696510,0.794118,0.742118,0.979439
7,0.007755,0.118684,0.715190,0.782007,0.747107,0.980424
8,0.003810,0.121702,0.714063,0.790657,0.750411,0.980227
9,0.003784,0.125179,0.714739,0.780277,0.746071,0.979965
10,0.003230,0.129150,0.705512,0.775087,0.738664,0.979636


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.21it/s]
There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.la

,experiment,shot_count,do_normalization,do_self_training,train_examples,pseudo_examples,validation_f1,test_f1
0,adapt_250,250,False,False,250,0,0.763006,0.643154


## 11. Main Experiments

The main experiment set is:
- source-only
- 250-shot adaptation
- 250-shot adaptation with normalization
- 250-shot adaptation with self-training
- 250-shot adaptation with normalization + self-training



In [11]:
# Main comparison set used in the notebook.
experiments = [
    # Required source-only baseline.
    {"name": "source_only", "shot_count": 0, "do_normalization": False, "do_self_training": False},
    # Plain few-shot adaptation.
    {"name": "adapt_250", "shot_count": 250, "do_normalization": False, "do_self_training": False},
    # Few-shot adaptation with normalization.
    {"name": "adapt_250_norm", "shot_count": 250, "do_normalization": True, "do_self_training": False},
    # Few-shot adaptation with self-training.
    {"name": "adapt_250_selftrain", "shot_count": 250, "do_normalization": False, "do_self_training": True},
    # Combined recipe: normalization plus self-training.
    {"name": "adapt_250_norm_selftrain", "shot_count": 250, "do_normalization": True, "do_self_training": True},
]

# Collect experiment summaries here.
results = []
# Run each experiment one by one.
for experiment in experiments:
    # Print progress in notebook output.
    print("Running", experiment["name"])
    # Execute experiment and store its summary.
    results.append(run_experiment(**experiment))

# Convert results into a table.
results_df = pd.DataFrame(results)
# Compute absolute gain relative to the source-only baseline.
results_df["gain_vs_source_only"] = results_df["test_f1"] - float(results_df.loc[results_df["experiment"] == "source_only", "test_f1"].iloc[0])
# Show experiments sorted by test F1.
results_df.sort_values("test_f1", ascending=False).round(4)


Running source_only


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 20639.63it/s]
RobertaForTokenClassification LOAD REPORT from: andi611/roberta-base-ner-conll2003
Key                             | Status     |                                                                                       
--------------------------------+------------+---------------------------------------------------------------------------------------
roberta.embeddings.position_ids | UNEXPECTED |                                                                                       
classifier.weight               | MISMATCH   | Reinit due to size mismatch - ckpt: torch.Size([9, 768]) vs model:torch.Size([7, 768])
classifier.bias                 | MISMATCH   | Reinit due to size mismatch - ckpt: torch.Size([9]) vs model:torch.Size([7])          

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISMATCH	:ckpt weights were loaded, but they did not m

Running adapt_250


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 15876.03it/s]
RobertaForTokenClassification LOAD REPORT from: andi611/roberta-base-ner-conll2003
Key                             | Status     |                                                                                       
--------------------------------+------------+---------------------------------------------------------------------------------------
roberta.embeddings.position_ids | UNEXPECTED |                                                                                       
classifier.weight               | MISMATCH   | Reinit due to size mismatch - ckpt: torch.Size([9, 768]) vs model:torch.Size([7, 768])
classifier.bias                 | MISMATCH   | Reinit due to size mismatch - ckpt: torch.Size([9]) vs model:torch.Size([7])          

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISMATCH	:ckpt weights were loaded, but they did not m

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.183691,0.094617,0.676516,0.752595,0.712531,0.978125
2,0.093299,0.084979,0.719472,0.754325,0.736486,0.980227
3,0.051757,0.093402,0.762653,0.756055,0.759340,0.982264
4,0.026189,0.099798,0.728707,0.799308,0.762376,0.981344
5,0.015369,0.119224,0.724307,0.768166,0.745592,0.980621
6,0.014905,0.119679,0.696510,0.794118,0.742118,0.979439
7,0.007755,0.118684,0.715190,0.782007,0.747107,0.980424
8,0.003810,0.121702,0.714063,0.790657,0.750411,0.980227
9,0.003784,0.125179,0.714739,0.780277,0.746071,0.979965
10,0.003230,0.129150,0.705512,0.775087,0.738664,0.979636


...

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.179679,0.094309,0.640867,0.716263,0.676471,0.975760
2,0.081878,0.086808,0.681745,0.730104,0.705096,0.978848
3,0.038552,0.110125,0.689034,0.728374,0.708158,0.978979
4,0.028212,0.113698,0.807767,0.719723,0.761208,0.982921
5,0.019652,0.120780,0.666667,0.775087,0.716800,0.977403
6,0.014237,0.120731,0.681600,0.737024,0.708229,0.978913
7,0.009338,0.115854,0.726384,0.771626,0.748322,0.981475
8,0.005463,0.120460,0.722930,0.785467,0.752902,0.981344
9,0.006306,0.125369,0.720517,0.771626,0.745196,0.981278
10,0.002408,0.124805,0.724756,0.769896,0.746644,0.981410


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.18it/s]
There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.la

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.030094,0.120653,0.782689,0.735294,0.758252,0.982198
2,0.018377,0.135430,0.772477,0.728374,0.749777,0.982132
3,0.010812,0.155151,0.782158,0.652249,0.711321,0.980293
4,0.011641,0.135633,0.775769,0.742215,0.758621,0.982067
5,0.005690,0.129616,0.765411,0.773356,0.769363,0.982921
6,0.005477,0.146530,0.756757,0.726644,0.741395,0.981607
7,0.001503,0.146879,0.762238,0.754325,0.758261,0.982395
8,0.001224,0.154310,0.749562,0.740484,0.744996,0.981672
9,0.001563,0.155151,0.759058,0.724913,0.741593,0.981607
10,0.002099,0.153319,0.775093,0.721453,0.747312,0.982132


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.19it/s]
There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.la

,experiment,shot_count,do_normalization,do_self_training,train_examples,pseudo_examples,validation_f1,test_f1,gain_vs_source_only
3,adapt_250_selftrain,250,False,True,250,250,0.7752,0.6722,0.1109
2,adapt_250_norm,250,True,False,250,0,0.7605,0.6563,0.0950
4,adapt_250_norm_selftrain,250,True,True,250,250,0.7694,0.6561,0.0947
1,adapt_250,250,False,False,250,0,0.7630,0.6432,0.0818
0,source_only,0,False,False,0,0,0.6948,0.5613,0.0000


## 12. Benchmark Against a Task-Specific WNUT Model

To put the budget-friendly adaptation results in context, we also evaluate a model already fine-tuned on WNUT: `emilys/twitter-roberta-base-WNUT`.

This benchmark is used as a reference point. Because it was trained on the full WNUT tag set, its predictions are remapped into the reduced label space used in this notebook. Unsupported predicted labels are treated as `O` in this reduced comparison.


In [12]:
# Mapping from full WNUT numeric ids to the reduced label space used here.
WNUT_ID_TO_REDUCED_ID = {
    # O stays O.
    0: LABEL2ID["O"],
    # Corporation becomes ORG.
    1: LABEL2ID["B-ORG"],
    2: LABEL2ID["I-ORG"],
    # Location becomes LOC.
    7: LABEL2ID["B-LOC"],
    8: LABEL2ID["I-LOC"],
    # Person becomes PER.
    9: LABEL2ID["B-PER"],
    10: LABEL2ID["I-PER"],
}


def map_benchmark_label_to_reduced_id(label_id: int) -> int:
    # Map full-WNUT prediction id into reduced label id.
    # Unsupported label types are treated as O for this reduced comparison.
    return WNUT_ID_TO_REDUCED_ID.get(int(label_id), LABEL2ID["O"])


def predict_with_benchmark_model(raw_split: Dataset) -> tuple[np.ndarray, np.ndarray]:
    # Load benchmark tokenizer.
    benchmark_tokenizer = AutoTokenizer.from_pretrained(BENCHMARK_MODEL_NAME, use_fast=True)
    # Load benchmark model already fine-tuned on full WNUT.
    benchmark_model = AutoModelForTokenClassification.from_pretrained(BENCHMARK_MODEL_NAME)
    # Use GPU if available.
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    # Move benchmark model to device.
    benchmark_model.to(device)
    # Switch benchmark model to evaluation mode.
    benchmark_model.eval()

    # Store predicted reduced-label ids.
    all_predictions = []
    # Store gold reduced-label ids.
    all_labels = []

    # Process one sentence at a time.
    for row in raw_split:
        # Tokenize the input sentence.
        encoded = benchmark_tokenizer(
            row["tokens"],
            is_split_into_words=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt",
        )
        # Recover mapping from subtokens to original word ids.
        word_ids = encoded.word_ids(batch_index=0)
        # Move tensors to the benchmark model device.
        encoded = {key: value.to(device) for key, value in encoded.items()}

        # Run benchmark inference.
        with torch.no_grad():
            logits = benchmark_model(**encoded).logits[0].cpu().numpy()

        # Predicted reduced ids for this sentence.
        reduced_prediction_ids = []
        # Gold reduced ids for this sentence.
        reduced_label_ids = []
        # Track processed words so we keep only the first subtoken.
        seen_word_ids = set()

        # Iterate over tokenized positions.
        for token_index, word_id in enumerate(word_ids):
            # Skip special tokens and repeated subtokens.
            if word_id is None or word_id in seen_word_ids:
                continue
            # Mark this word as processed.
            seen_word_ids.add(word_id)

            # Predict the full-WNUT label id for this word.
            predicted_label_id = int(np.argmax(logits[token_index]))
            # Map predicted full-WNUT label into reduced label space.
            reduced_prediction_ids.append(map_benchmark_label_to_reduced_id(predicted_label_id))
            # Keep the gold reduced target label.
            reduced_label_ids.append(int(row["target_ner_tags"][word_id]))

        # Save predictions for this sentence.
        all_predictions.append(reduced_prediction_ids)
        # Save gold labels for this sentence.
        all_labels.append(reduced_label_ids)

    # Pad sequences so they can be converted to arrays.
    max_len = max(len(row) for row in all_predictions)
    # Predicted ids use O as padding value.
    prediction_array = np.full((len(all_predictions), max_len), fill_value=0, dtype=int)
    # Gold labels use -100 so padded positions are ignored.
    label_array = np.full((len(all_labels), max_len), fill_value=-100, dtype=int)

    # Copy variable-length rows into fixed-size arrays.
    for i, (pred_row, label_row) in enumerate(zip(all_predictions, all_labels)):
        # Fill predicted ids.
        prediction_array[i, :len(pred_row)] = pred_row
        # Fill gold ids.
        label_array[i, :len(label_row)] = label_row

    # Return arrays compatible with the shared metric code.
    return prediction_array, label_array


# Run benchmark prediction on the reduced-label test set.
benchmark_predictions, benchmark_labels = predict_with_benchmark_model(test_raw)
# Compute reduced-label benchmark metrics.
benchmark_metrics = compute_metrics_from_arrays(benchmark_predictions, benchmark_labels)
# Store benchmark result in the same schema as the experiment table.
benchmark_result = {
    # Benchmark row name.
    "experiment": "benchmark_twitter_roberta_wnut",
    # No few-shot labeled target training in this notebook for the benchmark row.
    "shot_count": 0,
    # No normalization flag here.
    "do_normalization": False,
    # No self-training flag here.
    "do_self_training": False,
    # No gold few-shot training examples for the benchmark row itself.
    "train_examples": 0,
    # No pseudo-labeled examples for the benchmark row itself.
    "pseudo_examples": 0,
    # Validation is not computed for this external benchmark cell.
    "validation_f1": np.nan,
    # Main benchmark test F1.
    "test_f1": benchmark_metrics["f1"],
}

# Add the benchmark as a reference row next to notebook experiments.
comparison_df = pd.concat([results_df, pd.DataFrame([benchmark_result])], ignore_index=True)
# Compute gain over the source-only baseline for all rows, including the benchmark.
comparison_df["gain_vs_source_only"] = comparison_df["test_f1"] - float(comparison_df.loc[comparison_df["experiment"] == "source_only", "test_f1"].iloc[0])
# Show final comparison table.
comparison_df.sort_values("test_f1", ascending=False).round(4)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 17237.70it/s]
RobertaForTokenClassification LOAD REPORT from: emilys/twitter-roberta-base-WNUT
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,experiment,shot_count,do_normalization,do_self_training,train_examples,pseudo_examples,validation_f1,test_f1,gain_vs_source_only
5,benchmark_twitter_roberta_wnut,0,False,False,0,0,NaN,0.7053,0.1440
3,adapt_250_selftrain,250,False,True,250,250,0.7752,0.6722,0.1109
2,adapt_250_norm,250,True,False,250,0,0.7605,0.6563,0.0950
4,adapt_250_norm_selftrain,250,True,True,250,250,0.7694,0.6561,0.0947
1,adapt_250,250,False,False,250,0,0.7630,0.6432,0.0818
0,source_only,0,False,False,0,0,0.6948,0.5613,0.0000
